# K-Nearest Neighbours Example (Wine Quality Dataset)

Here it is demonstrated how to use the `KNNClassifier` and `KNNRegressor` modules from the CMOR-438 library.
In this example, the Wine Quality dataset is used to train, test, and evaluate both models.

**Goal: Classify wine quality tier and predict raw quality score from physicochemical features.**

Two tasks are demonstrated:
- **Classification:** Predict quality tier — Low (3–4), Mid (5–6), or High (7–8)
- **Regression:** Predict the raw continuous quality score (3–8)

## 1. Setup and Data Loading

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from k_nearest_neighbors import KNNClassifier, KNNRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv('../../../data/WineQT.csv').drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

Create both a 3-class classification target and a continuous regression target. Standardise features — KNN is sensitive to scale.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_reg = wine['quality'].values.astype(float)
y_clf = np.array([0 if q<=4 else (1 if q<=6 else 2) for q in y_reg])

X_tr, X_te, y_tr_clf, y_te_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42, stratify=y_clf)
_, __, y_tr_reg, y_te_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")
print(f"Class counts: {dict(zip(*np.unique(y_clf, return_counts=True)))}")

## 3. Sweep k Values

Rather than picking k arbitrarily, we sweep k from 1 to 20 and measure test performance for both models.
This is the standard way to tune KNN — choose k that maximises validation performance.

In [ ]:
k_range = range(1, 21)
clf_accs, reg_r2s = [], []
for k in k_range:
    kc = KNNClassifier(k=k, weights='distance').fit(X_tr, y_tr_clf)
    clf_accs.append(kc.accuracy(X_te, y_te_clf))
    kr = KNNRegressor(k=k, weights='distance').fit(X_tr, y_tr_reg)
    reg_r2s.append(kr.score(X_te, y_te_reg))

best_k_clf = list(k_range)[np.argmax(clf_accs)]
best_k_reg = list(k_range)[np.argmax(reg_r2s)]
print(f'Best K (Classifier): {best_k_clf}  Accuracy={max(clf_accs):.4f}')
print(f'Best K (Regressor):  {best_k_reg}  R²={max(reg_r2s):.4f}')

## 4. Results and Visualisation

Two plots are produced:
- **Classifier accuracy vs k** — shows the optimal k for the classification task; very small k overfits, very large k underfits
- **Regressor R² vs k** — same sweep for the regression task

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), clf_accs, 'o-', color='steelblue', lw=1.5, ms=5)
axes[0].axvline(best_k_clf, color='red', linestyle='--', lw=1.2, label=f'Best k={best_k_clf}')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('KNN Classifier - Accuracy vs k', fontweight='bold'); axes[0].legend()

axes[1].plot(list(k_range), reg_r2s, 'o-', color='darkorange', lw=1.5, ms=5)
axes[1].axvline(best_k_reg, color='red', linestyle='--', lw=1.2, label=f'Best k={best_k_reg}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('R²')
axes[1].set_title('KNN Regressor - R² vs k', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()